# Annexe A3 : pistes d'amélioration

**Place dans le projet.** Cette annexe documente la phase A3 de l'historique présenté dans
`notebooks/modelisation.ipynb` (section 4.2). Elle fait suite à l'annexe A2.

**Point de départ.** En A2, toutes les architectures plafonnent autour de R² HLC ≈ 0,44 en validation, et
la variabilité entre graines est faible. Avant d'accepter ce plafond, on teste trois idées qui pourraient
apporter de l'information au réseau :

| Idée | Hypothèse |
|---|---|
| 1. **Résidu à la réponse moyenne** | Une grande partie de T_int s'explique par le programme de chauffe, commun à tous les bâtiments. Retirer cette part et donner au réseau le résidu, signature propre au bâtiment, pourrait l'aider. |
| 2. **Pas de 2 min au lieu de 5** | Le moyennage à 5 min efface peut-être une dynamique rapide utile, notamment pour HLC_ext et τ_n. |
| 3. **GRU bidirectionnel** | Lire la série dans les deux sens permet d'interpréter le début de la mesure à la lumière de la suite (l'état initial des parois est inconnu). |

**Protocole.** Mêmes données (batchs 1 à 10), même découpage (`splits.npz` de A2) et mêmes hyperparamètres
(configuration Optuna de A2) : seule l'idée testée change. Un modèle de **contrôle**, identique à A2, sert
à mesurer l'écart dû au seul ré-entraînement.

**Exécution.** Ce notebook a tourné sur Kaggle (GPU Tesla T4) et n'est pas destiné à être ré-exécuté en
local. La copie téléchargée contient les sorties des premières cellules seulement. Les résultats de
toutes les variantes ont été sauvegardés dans `results/results_idees/` et sont relus en **section 9**.

## 1. Installation
Copie du paquet `coenergy` dans l'environnement Kaggle et repérage des données.

In [2]:
import os, sys, glob, shutil, zipfile, subprocess, importlib

CODE_DST = "/kaggle/working/coenergy-assessment"
NEW_MODULES = ["fe", "targets", "evaluation", "gbm", "nets", "ensemble", "predict", "runner"]

# Projet de base
base = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
assert base, "Dataset coenergy-code introuvable."
if not os.path.exists(CODE_DST):
    shutil.copytree(os.path.dirname(base[0]), CODE_DST)

# Nouveaux modules (patch)
patch = glob.glob("/kaggle/input/**/src/coenergy/runner.py", recursive=True)
if not patch:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            if any(n.endswith("src/coenergy/runner.py") for n in f.namelist()):
                f.extractall("/kaggle/working/patch")
    patch = glob.glob("/kaggle/working/patch/**/src/coenergy/runner.py", recursive=True)
assert patch, "Dataset coenergy-modelisation introuvable."
for m in NEW_MODULES:
    shutil.copy(os.path.join(os.path.dirname(patch[0]), m + ".py"),
                os.path.join(CODE_DST, "src", "coenergy", m + ".py"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DST, "--no-deps"], check=True)
src_dir = os.path.join(CODE_DST, "src")
if src_dir in sys.path:
    sys.path.remove(src_dir)
sys.path.insert(0, src_dir)
if CODE_DST not in sys.path:
    sys.path.insert(1, CODE_DST)
for name in list(sys.modules):
    if name == "coenergy" or name.startswith("coenergy."):
        del sys.modules[name]
importlib.invalidate_caches()
os.chdir(CODE_DST)

batches = sorted(glob.glob("/kaggle/input/**/batch_*.pt", recursive=True))
DATA_DIR = os.path.dirname(batches[0])
import coenergy, coenergy.runner, coenergy.nets
print("coenergy :", os.path.dirname(coenergy.__file__))
print("Données  :", DATA_DIR, f"({len(batches)} fichiers)")

coenergy : /kaggle/working/coenergy-assessment/src/coenergy
Données  : /kaggle/input/datasets/raraabdel/dataset (20 fichiers)


## 2. Configuration
On reprend exactement la meilleure configuration Optuna de A2 : CNN-GRU, 128 filtres, GRU 2 × 256, AdamW,
lr 4,2·10⁻⁴, décroissance cosinus. En A2, elle atteignait un R² moyen de validation de 0,2670 ± 0,0007
(3 graines) et R² HLC = 0,4387 : c'est la **référence à battre**.

In [3]:
import json, time, warnings, pkgutil, hashlib, traceback
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from scipy.signal import lfilter

from coenergy.config import DataConfig
from coenergy.data import load_simulations
from coenergy.evaluation import ExperimentLog, Timer, indicator_metrics, load_splits
from coenergy.ensemble import WeightedEnsemble, cross_validated_predictions
from coenergy.nets import TorchSettings, SeqEncoder, HybridNet, fit_torch, predict_torch
from coenergy.runner import ExperimentContext, r2_mean

warnings.filterwarnings("ignore", category=UserWarning)
ROOT = Path(CODE_DST)
OUT = Path("/kaggle/working/results_idees")
(OUT / "poids").mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PARAM = "A"

# Configuration Optuna de cette nuit (référence : R² val moyen 0,2670 ± 0,0007, HLC 0,4387)
OPT_ENC = dict(kind="cnn_gru", channels=128, n_conv=2, kernel=3, gru_hidden=256, gru_layers=2, dropout=0.01356818667316142)
OPT_S = TorchSettings(lr=0.00041702416368727104, weight_decay=0.0020651425578959264, batch_size=1024,
                      epochs=80, patience=12, optimizer="adamw", scheduler="cosine", loss="mse", seed=42)
TEST_RESOLUTION_2MIN = True     # idée 2 (la plus longue : ~1 h)
print("Device :", DEVICE)

Device : cuda


## 3. Données et découpage identique à A2
Le découpage est **relu** depuis `results_dl/splits.npz`, et non recalculé. Les groupes de motifs sont
reconstruits par empreinte (hachage) de U, et une assertion vérifie la cohérence avec A2. Tous les
scores de cette annexe sont ainsi directement comparables à ceux de A2. Sortie attendue : 400 968
simulations en train, 57 290 en validation, 41 742 en test, 96 pas de temps.

In [4]:
data = load_simulations(DATA_DIR, util_dir=ROOT, n_batches=10, n_per_batch=None, downsample=5, seed=42)
LABELS = data.indicator_labels
Y = data.indicators().numpy().astype(np.float64)
T, U = data.temperature.numpy().astype(np.float64), data.excitation.numpy().astype(np.float64)
splits = load_splits(glob.glob("/kaggle/input/**/results_dl/splits.npz", recursive=True)[0])
splits = {k: splits[k] for k in ("train", "val", "test")}
keys = np.array([hashlib.blake2b(r.tobytes(), digest_size=8).hexdigest() for r in np.ascontiguousarray(U)])
groups = pd.factorize(keys)[0]
assert not set(groups[splits["train"]]) & set(groups[splits["test"]]), "Découpage incohérent avec cette nuit"
print({k: len(v) for k, v in splits.items()}, "| L =", T.shape[1])

log = ExperimentLog(OUT / "experiences_idees.csv", LABELS)
ctx = ExperimentContext(Y, LABELS, splits, groups, pd.DataFrame(index=np.arange(len(Y))), T, U,
                        data.dt_minutes, log, OUT, device=DEVICE)

def safe(label, fn, *a, **k):
    try:
        return fn(*a, **k)
    except Exception as e:
        print(f"ÉCHEC « {label} » : {type(e).__name__}: {e}")
        open(OUT / "erreurs.log", "a").write(f"{label}\n{traceback.format_exc()}\n")
        torch.cuda.empty_cache()

batch_1.pt : 50000 simulations conservées
batch_2.pt : 50000 simulations conservées
batch_3.pt : 50000 simulations conservées
batch_4.pt : 50000 simulations conservées
batch_5.pt : 50000 simulations conservées
batch_6.pt : 50000 simulations conservées
batch_7.pt : 50000 simulations conservées
batch_8.pt : 50000 simulations conservées
batch_9.pt : 50000 simulations conservées
batch_10.pt : 50000 simulations conservées
{'train': 400968, 'val': 57290, 'test': 41742} | L = 96


## 4. Idée 1 : la réponse moyenne de la population

On ajuste sur le train un modèle **linéaire commun à tous les bâtiments** :

$$\mathbb{E}[\Delta T(t) \mid U] = \sum_{k=0}^{t} h[k]\,u(t-k) + f[t]$$

- $h$ est la **réponse impulsionnelle moyenne** : comment, en moyenne, 1 kW injecté à l'instant $t-k$
  élève la température à l'instant $t$ ;
- $f[t]$ est la **dérive moyenne** en l'absence de chauffage.

L'ajustement se fait par moindres carrés régularisés (ridge) sur 60 000 simulations du train, en
accumulant les équations normales pas de temps par pas de temps pour limiter la mémoire.

Le **résidu** $R(t) = \Delta T(t) - \mathbb{E}[\Delta T(t) \mid U]$ est ce qui distingue ce bâtiment du
bâtiment moyen. La cellule affiche la part de variance de ΔT expliquée par cette réponse commune
(sur la validation) : elle mesure à quel point le programme de chauffe, et non le bâtiment, détermine
la courbe de température.

In [ ]:
def fit_population_response(dT, u_kw, idx, n_sub=60_000, ridge=1e-4, seed=0):
    """Ajuste E[ΔT(t) | U] = Σ_k h[k]·u(t-k) + f[t] par moindres carrés sur le train."""
    sub = np.random.default_rng(seed).choice(idx, size=min(n_sub, len(idx)), replace=False)
    y, u = dT[sub], u_kw[sub]
    L = y.shape[1]
    xtx, xty = np.zeros((2 * L, 2 * L)), np.zeros(2 * L)
    for t in range(L):
        A = np.zeros((len(sub), L))
        A[:, : t + 1] = u[:, t::-1]                       # u(t), u(t-1), …, u(0)
        s = A.sum(axis=0)
        xtx[:L, :L] += A.T @ A
        xtx[:L, L + t] += s
        xtx[L + t, :L] += s
        xtx[L + t, L + t] += len(sub)
        xty[:L] += A.T @ y[:, t]
        xty[L + t] += y[:, t].sum()
    reg = ridge * np.trace(xtx) / (2 * L) * np.eye(2 * L)
    theta = np.linalg.solve(xtx + reg, xty)
    return theta[:L], theta[L:]

def population_prediction(u_kw, h, f):
    return lfilter(h, [1.0], u_kw, axis=1) + f[None, :]

dT = T - T[:, :1]
u_kw = U / 1000.0
with Timer() as t:
    h_pop, f_pop = fit_population_response(dT, u_kw, splits["train"])
    mean_resp = population_prediction(u_kw, h_pop, f_pop)
R = dT - mean_resp                                            # résidu : signature propre au bâtiment
v = splits["val"]
share = 1 - ((dT[v] - mean_resp[v]) ** 2).sum() / ((dT[v] - dT[v].mean(0)) ** 2).sum()
print(f"Réponse moyenne ajustée en {t.seconds:.0f} s | part de la variance de ΔT expliquée (val) : {share:.3f}")
print(f"Écart-type du résidu : {R[v].std():.3f} K (contre {dT[v].std():.3f} K pour ΔT)")
np.savez(OUT / "reponse_population.npz", h=h_pop, f=f_pop)

## 5. Outils : encodeur bidirectionnel et entraînement des variantes

- `standardize` : empile les canaux choisis et les standardise avec les statistiques du **train** seulement.
- `BiGRUEncoder` : mêmes convolutions que le CNN-GRU, puis un GRU **bidirectionnel**. La représentation
  concatène les états finaux des deux sens et la moyenne temporelle.
- `train_variant` : entraîne une variante avec la configuration Optuna, enregistre ses scores dans le
  journal (`experiences_idees.csv`) et sauvegarde ses poids (`poids/<nom>.pt`).

In [ ]:
def standardize(channels, idx):
    x = np.stack(channels, axis=1).astype(np.float32)          # (N, C, L)
    m, s = x[idx].mean(axis=(0, 2)), x[idx].std(axis=(0, 2)) + 1e-6
    return (x - m[None, :, None]) / s[None, :, None]

class BiGRUEncoder(nn.Module):
    """Convolutions puis GRU bidirectionnel (lecture dans les deux sens)."""
    def __init__(self, in_ch, channels=128, n_conv=2, kernel=3, hidden=256, layers=2, dropout=0.0):
        super().__init__()
        blocks, c = [], in_ch
        for _ in range(n_conv):
            blocks += [nn.Conv1d(c, channels, kernel, padding=kernel // 2), nn.BatchNorm1d(channels), nn.GELU(), nn.MaxPool1d(2)]
            c = channels
        self.conv = nn.Sequential(*blocks)
        self.gru = nn.GRU(channels, hidden, layers, batch_first=True, bidirectional=True,
                          dropout=dropout if layers > 1 else 0.0)
        self.out_features = 4 * hidden
    def forward(self, x):
        out, h_n = self.gru(self.conv(x).transpose(1, 2))
        return torch.cat([h_n[-2], h_n[-1], out.mean(dim=1)], dim=1)

def make_net(in_ch, bidirectional=False):
    if bidirectional:
        enc = BiGRUEncoder(in_ch, 128, 2, 3, 256, 2, OPT_ENC["dropout"])
    else:
        enc = SeqEncoder(OPT_ENC["kind"], in_ch, OPT_ENC["channels"], OPT_ENC["n_conv"], OPT_ENC["kernel"],
                         OPT_ENC["gru_hidden"], OPT_ENC["gru_layers"], OPT_ENC["dropout"])
    return HybridNet(enc, 0, 64, 64, 5, OPT_ENC["dropout"])

def train_variant(name, X, in_ch, note, bidirectional=False, s=OPT_S, splits_=None, y_=None):
    sp = splits_ or splits
    yy = Y if y_ is None else y_
    tt = ctx.target(PARAM)
    sets = {"train": sp["train"], "train_eval": ctx.train_eval, "val": sp["val"], "test": sp["test"]}
    xs = {k: torch.from_numpy(X[v]).to(DEVICE) for k, v in sets.items()}
    z = {k: torch.from_numpy(tt.encode(yy[sets[k]])).to(DEVICE) for k in ("train", "val")}
    torch.manual_seed(s.seed)
    net = make_net(in_ch, bidirectional).to(DEVICE)
    with Timer() as tm:
        info = fit_torch(net, (xs["train"],), z["train"], (xs["val"],), z["val"], s)
    preds = {k: tt.decode(predict_torch(net, (xs[k],))) for k in ("train_eval", "val", "test")}
    ctx.record(name, "idée (PyTorch)", {"variante": note, "in_ch": in_ch, "bidirectionnel": bidirectional,
                                        **asdict(s)}, preds, tm.seconds, None, best_epoch=info["best_epoch"] + 1)
    torch.save(net.state_dict(), OUT / "poids" / f"{name}.pt")
    del xs
    torch.cuda.empty_cache()

## 6. Variantes testées (pas de 5 min)

| Variante | Canaux d'entrée | GRU | Rôle |
|---|---|---|---|
| `reference_T_dT_U` | T, ΔT, U | simple | **Contrôle** : identique à A2, mesure l'écart dû au ré-entraînement |
| `residu_4canaux` | T, ΔT, U, **R** | simple | Idée 1 : le résidu en plus des entrées habituelles |
| `residu_2canaux` | **R**, U | simple | Idée 1 bis : le résidu à la place de T |
| `bigru` | T, ΔT, U | **bidirectionnel** | Idée 3 |
| `bigru_residu` | T, ΔT, U, **R** | **bidirectionnel** | Idées 1 et 3 combinées |

In [ ]:
tr = splits["train"]
VARIANTS = {
    # nom : (canaux, description, bidirectionnel)
    "reference_T_dT_U":  ([T, dT, u_kw],     "référence : mêmes entrées que cette nuit (contrôle)", False),
    "residu_4canaux":    ([T, dT, u_kw, R],  "idée 1 : + résidu à la réponse moyenne", False),
    "residu_2canaux":    ([R, u_kw],         "idée 1 bis : résidu et U seulement", False),
    "bigru_residu":      ([T, dT, u_kw, R],  "idées 1 + 3 : résidu + GRU bidirectionnel", True),
    "bigru":             ([T, dT, u_kw],     "idée 3 : GRU bidirectionnel", False | True),
}
for name, (chans, note, bidir) in VARIANTS.items():
    X = standardize(chans, tr)
    safe(name, train_variant, name, X, X.shape[1], note, bidir)
    del X
log.table()[["nom", "r2_train_mean", "r2_val_mean", "ecart_r2_train_val", "r2_val_HLC", "r2_val_HLC_mit",
             "r2_val_HLC_ext", "r2_val_tau_b", "r2_val_tau_n", "r2_val_tau_inf", "temps_s"]].round(4)

## 7. Idée 2 : pas de 2 minutes
Même modèle et mêmes entrées que le contrôle, mais les séries sont moyennées sur 2 min au lieu de 5
(240 points au lieu de 96). Une assertion vérifie que les simulations sont dans le même ordre. Ce modèle
(`resolution_2min.pt`) est réutilisé tel quel dans le modèle final, sous le nom `res2_g42_10batchs`.

In [ ]:
if TEST_RESOLUTION_2MIN:
    def run_2min():
        d2 = load_simulations(DATA_DIR, util_dir=ROOT, n_batches=10, n_per_batch=None, downsample=2, seed=42)
        T2 = d2.temperature.numpy().astype(np.float64)
        U2 = d2.excitation.numpy().astype(np.float64) / 1000.0
        assert np.allclose(d2.indicators().numpy(), Y, atol=1e-4), "ordre des simulations différent"
        X2 = standardize([T2, T2 - T2[:, :1], U2], tr)
        print("Résolution 2 min : L =", X2.shape[2])
        train_variant("resolution_2min", X2, 3, "idée 2 : pas de 2 min (240 points)", False)
    safe("résolution 2 min", run_2min)
log.table().iloc[:, :12]

## 8. Comparaison avec A2 et ensemble
- Tableau de **validation** des variantes, à côté du meilleur modèle Optuna et de l'ensemble de A2
  (`comparaison_idees.csv`).
- **Ensemble** : on vérifie si ajouter les nouvelles variantes aux modèles de A2 améliore la prédiction
  hors pli, c'est-à-dire si elles apportent une information différente de celle des modèles existants.

In [ ]:
night = np.load(glob.glob("/kaggle/input/**/results_dl/predictions.npz", recursive=True)[0])
night_val = {k[5:]: night[k] for k in night.files if k.startswith("val__")}
night_test = {k[6:]: night[k] for k in night.files if k.startswith("test__")}
yv, yt = Y[splits["val"]], Y[splits["test"]]

# Tableau : variantes du jour + références de la nuit (même découpage)
rows = {}
for n, p in {**{f"nuit_{k}": v for k, v in night_val.items() if k in ("optuna_best", "ensemble_pondere_dl")},
             **{n: ctx.preds_val[n] for n in ctx.preds_val}}.items():
    r2 = indicator_metrics(yv, p, LABELS)["r2"]
    rows[n] = {"R² val moyen": r2.mean(), **{f"R² val {l}": r2[l] for l in LABELS}}
comp = pd.DataFrame(rows).T.sort_values("R² val HLC", ascending=False)
comp.to_csv(OUT / "comparaison_idees.csv")
display(comp.round(4))

# Ensemble : modèles de la nuit + nouvelles variantes (poids appris sur la validation, score hors pli)
pool_val = {**{f"nuit_{k}": v for k, v in night_val.items() if k in
               ("optuna_best", "cnn_gru", "tcn", "hybride_cnn_gru", "hybride_tcn")},
            **{n: ctx.preds_val[n] for n in ctx.preds_val}}
pool_test = {**{f"nuit_{k}": night_test[k] for k in night_val if f"nuit_{k}" in pool_val},
             **{n: ctx.preds_test[n] for n in ctx.preds_test}}
for label, names in [("nuit seule", [n for n in pool_val if n.startswith("nuit_")]),
                     ("nuit + idées", list(pool_val))]:
    pv = {n: pool_val[n] for n in names}
    oof = cross_validated_predictions(WeightedEnsemble, pv, yv, groups[splits["val"]], n_splits=5)
    comb = WeightedEnsemble().fit(pv, yv)
    r_val = indicator_metrics(yv, oof, LABELS)["r2"]
    print(f"Ensemble {label:13s} | R² val (hors pli) HLC = {r_val['HLC']:.4f}, moyen = {r_val.mean():.4f}")
    if label == "nuit + idées":
        ENS = (names, comb)

## 9. Résultats de l'exécution (relus depuis `results/results_idees/`)
Ces cellules s'exécutent **en local**, sans GPU ni données.

In [4]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 200, "display.max_columns", 30)

def find_dir(name: str) -> Path | None:
    """Cherche results/<name> depuis notebooks/annexes/, notebooks/ ou la racine du projet."""
    for base in (Path("../../results"), Path("../results"), Path("results")):
        if (base / name).exists():
            return base / name
    print(f"Dossier results/{name} introuvable : téléchargez-le depuis l'onglet Output du notebook Kaggle.")
    return None

def show_csv(directory: Path | None, filename: str, **kwargs) -> pd.DataFrame | None:
    """Affiche un CSV de résultats s'il existe."""
    if directory is None or not (directory / filename).exists():
        print(f"(fichier absent : {filename})")
        return None
    df = pd.read_csv(directory / filename, **kwargs)
    display(df.round(4))
    return df

RES = find_dir("results_idees")
print("Dossier :", RES)

Dossier : ..\..\results\results_idees


### 9.1 Variantes comparées aux références de A2 (validation)

In [5]:
show_csv(RES, "comparaison_idees.csv", index_col=0) if RES else None

,R² val moyen,R² val HLC,R² val HLC_mit,R² val HLC_ext,R² val tau_b,R² val tau_n,R² val tau_inf
resolution_2min,0.2780,0.4469,0.2796,0.4655,0.1192,0.2659,0.0907
nuit_ensemble_pondere_dl,0.2755,0.4456,0.2812,0.4601,0.1153,0.2627,0.0882
nuit_optuna_best,0.2688,0.4387,0.2770,0.4508,0.1088,0.2549,0.0825
reference_T_dT_U,0.2708,0.4386,0.2759,0.4494,0.1146,0.2574,0.0887
residu_4canaux,0.2645,0.4351,0.2774,0.4277,0.1099,0.2520,0.0850
bigru,0.2651,0.4337,0.2753,0.4378,0.1083,0.2542,0.0816
bigru_residu,0.2607,0.4336,0.2753,0.4280,0.0997,0.2521,0.0756
residu_2canaux,0.2590,0.4312,0.2757,0.4211,0.1004,0.2483,0.0770


,R² val moyen,R² val HLC,R² val HLC_mit,R² val HLC_ext,R² val tau_b,R² val tau_n,R² val tau_inf
resolution_2min,0.277972,0.446922,0.279625,0.465463,0.119198,0.265913,0.090710
nuit_ensemble_pondere_dl,0.275539,0.445603,0.281175,0.460134,0.115342,0.262744,0.088239
nuit_optuna_best,0.268790,0.438695,0.276999,0.450815,0.108777,0.254937,0.082517
reference_T_dT_U,0.270768,0.438576,0.275853,0.449438,0.114604,0.257426,0.088709
residu_4canaux,0.264519,0.435118,0.277448,0.427691,0.109884,0.251999,0.084975
bigru,0.265147,0.433747,0.275261,0.437780,0.108295,0.254187,0.081611
bigru_residu,0.260736,0.433605,0.275315,0.428031,0.099686,0.252129,0.075650
residu_2canaux,0.258952,0.431217,0.275685,0.421126,0.100400,0.248265,0.077019


### 9.2 Journal détaillé (R² train et validation, temps de calcul)

In [6]:
if RES and (RES / "experiences_idees.csv").exists():
    j = pd.read_csv(RES / "experiences_idees.csv")
    cols = [c for c in ["nom", "r2_train_mean", "r2_val_mean", "ecart_r2_train_val", "r2_val_HLC",
                        "r2_val_HLC_mit", "r2_val_HLC_ext", "r2_val_tau_b", "r2_val_tau_n",
                        "r2_val_tau_inf", "temps_s"] if c in j.columns]
    display(j[cols].round(4))

,nom,r2_train_mean,r2_val_mean,ecart_r2_train_val,r2_val_HLC,r2_val_HLC_mit,r2_val_HLC_ext,r2_val_tau_b,r2_val_tau_n,r2_val_tau_inf,temps_s
0,reference_T_dT_U,0.2949,0.2708,0.0241,0.4386,0.2759,0.4494,0.1146,0.2574,0.0887,1701.7
1,residu_4canaux,0.2894,0.2645,0.0249,0.4351,0.2774,0.4277,0.1099,0.2520,0.0850,1178.1
2,residu_2canaux,0.2816,0.2590,0.0227,0.4312,0.2757,0.4211,0.1004,0.2483,0.0770,979.2
3,bigru_residu,0.2770,0.2607,0.0162,0.4336,0.2753,0.4280,0.0997,0.2521,0.0756,1716.0
4,bigru,0.2925,0.2651,0.0273,0.4337,0.2753,0.4378,0.1083,0.2542,0.0816,2787.5
5,resolution_2min,0.2987,0.2780,0.0207,0.4469,0.2796,0.4655,0.1192,0.2659,0.0907,4939.5


## 10. Conclusions de la phase A3

1. **Pas de 2 min : seule idée gagnante.** Le CNN-GRU au pas de 2 min atteint R² HLC = 0,447 en
   validation, contre 0,439 pour le meilleur modèle Optuna de A2 au pas de 5 min, soit **+0,008**. Les
   chiffres ont été recalculés sur les mêmes simulations dans `modelisation.ipynb`, section 5.1. Le
   moyennage à 5 min effaçait donc une partie de la dynamique rapide. **Décision : pas de 2 min pour le
   modèle final.**
2. **Résidu à la réponse moyenne : pas de gain.** R² HLC de 0,435 (4 canaux) et 0,431 (2 canaux), contre
   0,439 pour le contrôle. Le résidu est une transformation linéaire de T et U, que le réseau sait déjà
   apprendre ; il n'apporte pas d'information nouvelle.
3. **GRU bidirectionnel : pas de gain.** R² HLC de 0,434 (0,434 avec le résidu), contre 0,439 pour le
   contrôle. Relire la série à l'envers ne compense pas l'inconnue principale, l'état thermique initial
   des parois.
4. Ces résultats négatifs **confirment le plafond d'information** mis en évidence dans l'EDA : la limite
   tient à ce que contiennent 8 h de mesure, pas à l'architecture. Le levier restant était la
   **quantité de données**, exploitée dans la phase finale (20 batchs au lieu de 10).

Le contrôle (`reference_T_dT_U`, 0,439) retrouve exactement le score du meilleur modèle Optuna de A2
(0,439) : les écarts mesurés sont donc dus aux idées testées, et non au ré-entraînement. Scores détaillés :
section 9.